## MLS SS25 Exercise 3: Find Suspicious Users (31P)
*Adapted from an exercise created by Dennis Eisermann*

Unsupervised learning techniques identify hidden patterns in unlabeled data. In contrast to supervised approaches, they do not rely on predetermined target variables (i.e., classes). This approach can cluster users by their behavior and thus detect potentially suspicious behavior.
You use the [User-Computer Authentication Associations in Time](https://csr.lanl.gov/data/auth/) dataset, which contains 708,304,516 successful authentication events over nine months. 11,362 unique users (Us) perform authenticationss at 22,284 unique computers (Cs). Timestamps are seconds-based offsets from an anonymized epoch. Get the first [30-day file](https://bwsyncandshare.kit.edu/s/4NCeo5KEDYnBTcj) `lanl-auth-dataset-1-00.bz2` (*or* after registering on the website of the authors above) to start experimenting.

### Setup

Install the necessary libraries to perform the following tasks.

In [3]:
!pip3 install numpy
!pip3 install matplotlib
!pip3 install scikit-learn
!pip3 install kneed
!pip3 install pandas

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


### Task 1 – Exploration of the Dataset (11P)

To get familiar with the dataset and build an initial understanding, try to find correlations and biases in the measured data.

1. Load the csv from `lanl-auth-dataset-1-00.bz2` in a dataframe. Convert the `Time` field to [datetime](https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html) and print the result. (2P)

In [4]:
import pandas as pd

# Read Dataset
column_names = [ 'timestamp', 'user', 'computer' ]
df_UCAAT = pd.read_csv(
    '../data/lanl-auth-dataset-1-00.bz2',
    compression='bz2',
    header=None,
    names=column_names,
)

df_UCAAT['timestamp'] = pd.to_datetime(df_UCAAT['timestamp'], unit='s')

print(df_UCAAT)

                   timestamp  user computer
0        1970-01-01 00:00:01    U1       C1
1        1970-01-01 00:00:01    U1       C2
2        1970-01-01 00:00:02    U2       C3
3        1970-01-01 00:00:03    U3       C4
4        1970-01-01 00:00:06    U4       C5
...                      ...   ...      ...
67054835 1970-01-29 00:00:00  U189    C1450
67054836 1970-01-29 00:00:00  U104     C126
67054837 1970-01-29 00:00:00  U104     C127
67054838 1970-01-29 00:00:00   U12     C210
67054839 1970-01-29 00:00:00  U112      C82

[67054840 rows x 3 columns]


2. Show the minimal and maximal time in the dataset. (1P)

In [5]:
print("Min time:", df_UCAAT['timestamp'].min())
print("Max time:", df_UCAAT['timestamp'].max())

Min time: 1970-01-01 00:00:01
Max time: 1970-01-29 00:00:00


3. pandas provides a lot of useful built-in exporation methods. "Describe" how often users create events, i.e., calculate basic statistics like count, mean, and quantiles. (1P)

In [6]:
count = df_UCAAT.shape[0]
print("Total number of records:", count)
mean = df_UCAAT['timestamp'].mean()
print("Mean time:", mean)
quantiles = df_UCAAT['timestamp'].quantile([0.25, 0.5, 0.75])
print("Quantiles:", quantiles)

Total number of records: 67054840
Mean time: 1970-01-14 20:25:40.541464778
Quantiles: 0.25   1970-01-07 22:54:00.000
0.50   1970-01-15 01:35:00.500
0.75   1970-01-23 00:14:21.000
Name: timestamp, dtype: datetime64[ns]


4. Find out which are the five most frequent users. (1P)

In [7]:
histogram = df_UCAAT['user'].value_counts()

print(histogram[:5])

user
U12      2676710
U13      1194429
U128      900805
U4148     796339
U60       696906
Name: count, dtype: int64


5. Calculate the number of logins separately for each computer. Print on how many different computers the user U12 logged into? (2P)

In [8]:

histogram = df_UCAAT[df_UCAAT['user'] == 'U12']['computer'].value_counts().shape[0]
print(histogram)

164


6. Think about typical user login patterns you could imagine. Find one argument in support and one against whether U12 is a suspicious user. (2P)

Argument for typical user:
 - Maintenance/Testing user could be possible

Argument against typical user:
- Typical user does not use 164 PCs

7. Get the maximum number of times U12 has logged in [within](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html) one minute. Use a pandas built-in method to solve this. (1P)

In [9]:
df_U12 = df_UCAAT[df_UCAAT['user'] == 'U12']

max_logins_per_minute = df_U12.rolling('min', on='timestamp').count()['user'].max()

print(df_U12.count())

timestamp    2676710
user         2676710
computer     2676710
dtype: int64


8. Calculate how often U12 has logged into different computers within one minute by grouping the dataframe before checking each timeframe. (1P)

In [11]:
# Filtere User
df_U12 = df_UCAAT[df_UCAAT['user'] == 'U12'].copy()

# Nach Zeit sortieren und als Index setzen
df_U12 = df_U12.sort_values('timestamp').set_index('timestamp')

# Rolling-Window: 1 Minute rückblickend
rolling_window = df_U12['computer'].rolling('1min')

# 1️⃣ Anzahl eindeutiger Computer im letzten 1-Minuten-Fenster
df_U12['unique_pc_count'] = rolling_window.apply(lambda x: x.nunique(), raw=False)

# 2️⃣ Welche Computer im letzten 1-Minuten-Fenster?
df_U12['pcs_in_window'] = rolling_window.apply(lambda x: ','.join(sorted(set(x))), raw=False)

print(df_U12)


DataError: No numeric types to aggregate

### Task 2 – Feature Engineering (6P)

In this task you create a dataset with new features to support the model in analysing the user behavior. You try to differentiate between normal users and suspicious ones with unusual behavior.


1. Build a new empty dataset. You will use it to store the relevant features to describe each user. (1P)

In [12]:
import pandas as pd

new_df = pd.DataFrame()

2. Add the number of login events for each user to the dataset. Name the column `login_numbers`. (1P)

In [13]:
new_df['login_numbers'] = df_UCAAT.groupby('user').size()
print(new_df['login_numbers'])

user
U1       83793
U10      44533
U100     15322
U1000    95553
U1001     2423
         ...  
U995      2786
U996      3678
U997     21162
U998      2215
U999      3195
Name: login_numbers, Length: 9924, dtype: int64


3. Add the number of different computers for each user. Name the column `computer_numbers`. (1P)

In [14]:
new_df['computer_numbers'] = df_UCAAT.groupby('user')['computer'].nunique()
print(new_df['computer_numbers'])

user
U1       39
U10      18
U100     15
U1000    23
U1001     7
         ..
U995     10
U996      8
U997     74
U998      5
U999     11
Name: computer_numbers, Length: 9924, dtype: int64


4. Add max logins within a minute for each user. Name the column `max_logins_in_1min`. This will help you find suspicious users with frequent logins in a short amount of time. (1P)

In [ ]:
def get_max_logins_in_1min(group):
    group = group.sort_values("timestamp")
    group = group.set_index("timestamp")
    
    # Count in Sliding Window
    rolling_counts = group["user"].rolling("1min").count()

    return rolling_counts.max()

new_df["max_logins_in_1min"] = df_UCAAT.groupby("user").apply(get_max_logins_in_1min)
print(new_df['max_logins_in_1min'])



user
U1        84.0
U10      112.0
U100      32.0
U1000     94.0
U1001     26.0
         ...  
U995      21.0
U996      28.0
U997      32.0
U998      15.0
U999      26.0
Name: max_logins_in_1min, Length: 9924, dtype: float64


/var/folders/s2/mw78n4f94wq0f0fnktg6j_500000gn/T/ipykernel_8926/3986448229.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df["max_logins_in_1min"] = df_UCAAT.groupby("user").apply(get_max_logins_in_1min)


5. Add to the dataset how many times each user has a login event outside the usual business hours between 6:00h and 18:00h. Filter out the weekend. Remember to check for undefined values and handle them. Call the column `out_of_business_times`. We assume that the dataset starts on midnight. (2P)

In [ ]:
from datetime import time

new_df['out_of_business_hours'] = df_UCAAT.groupby('user').apply(
    lambda group: group[
        (group['timestamp'].dt.time < time(6, 0)) | (group['timestamp'].dt.time > time(18, 0))
    ].shape[0]
    )
    


/var/folders/s2/mw78n4f94wq0f0fnktg6j_500000gn/T/ipykernel_8926/3440079565.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df['out_of_business_hours'] = df_UCAAT.groupby('user').apply(


In [17]:
print(new_df['out_of_business_hours'])

user
U1       40110
U10       9122
U100      6109
U1000    44713
U1001      656
         ...  
U995       525
U996      1707
U997      7816
U998       852
U999      1316
Name: out_of_business_hours, Length: 9924, dtype: int64


### Task 3 – Model Training and Evaluation (14P)

In this task you are using a model to cluster the data. This should allow you to identify potentially suspicious users.

1. Define a pipeline which [standard scales](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) its inputs and use [KMeans++](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) afterwards. Use six clusters and 42 as random state. (1P)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


2. Cluster the users based on `out_of_business_times` and `max_logins_in_1m`. Visualize the user values and the centroids as [scatter plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html). (3P)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt


3. Calculate the kMeans++ error (with all features) for 1 to 10 clusters. Use the [elbow method](https://www.geeksforgeeks.org/elbow-method-for-optimal-value-of-k-in-kmeans/) to determine the optimal number of clusters. Plot your results. You can use the [kneedle libary](https://pypi.org/project/kneed/). (3P)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kneed import KneeLocator


4. Use KMeans++ with the optimal numbers of clusters you determined above. Calculate the outliers based on the centroids. Outliers are datapoints with a higher standard deviation than the average data. Print the outliers as table and the percentage of outliers in comparison to the dataset. (3P)

In [ ]:

print(outliers)
print(f"Outliers: {percentage_outliers}%")

5. Print the strongest outlier. (1P)

6. Provide one argument in favor and one against the conclusion that the determined outlier is a malicious user. (2P)

*your answer*

7. Consider a realistic business scenario and provide one strategy, how to confirm an outlier as malicious actor? (1P)

*your answer*